<a href="https://colab.research.google.com/github/wolayav/2044-numpy-analisis-numerico-eficiente-con-python/blob/main/Proyecto2_WillyOlaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Configuración de modelo y documento
Como hemos observado en la clase 2 de Alura, configuramos el modelo de gemini 2.5-flash y cargaremos el documento que nos servirá para nuestro caso de uso.

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
from google import genai

cliente = genai.Client()

In [ ]:
respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents="Mejores métodos de enseñanza para estudiantes de secundaria 2026"
)

print(respuesta.text)

Para los estudiantes de secundaria que se graduarán en 2026 (es decir, los que están actualmente en los últimos años de secundaria), los métodos de enseñanza más efectivos son aquellos que los preparan no solo académicamente, sino también para un mundo en constante cambio, donde la adaptabilidad, el pensamiento crítico, la colaboración y la alfabetización digital son fundamentales.

Aquí te presento una combinación de los "mejores métodos de enseñanza" que son relevantes y eficaces para esta generación:

### Métodos de Enseñanza Centrados en el Estudiante y Activos

1.  **Aprendizaje Basado en Proyectos (ABP/PBL - Project-Based Learning):**
    *   **Descripción:** Los estudiantes trabajan en un proyecto extendido para resolver un problema real, responder a una pregunta compleja o crear un producto significativo. Integran conocimientos de diversas asignaturas.
    *   **Por qué es efectivo:** Fomenta el pensamiento crítico, la creatividad, la resolución de problemas, la colaboración, l

In [ ]:
from google.colab import files

os.makedirs("PDFs",exist_ok=True)
uploader = files.upload()
for archivo in uploader.keys():
  os.rename(archivo,f"PDFs/{archivo}")

Saving Estrategias y Metodologías Pedagógicas.pdf to Estrategias y Metodologías Pedagógicas.pdf


# Preparando el RAG
Visto en clase, se usa la misma estrucutra, con la diferencia de usar solo un vectorstore ya que la longitud del fragmento no supera el límite gratuito.

In [ ]:
!pip install -q langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
documentos = []

for archivo2 in os.listdir("PDFs"):
  ruta = os.path.join("PDFs",archivo2)
  loader = PyPDFLoader(ruta)
  paginas = loader.load()
  documentos.extend(paginas)

documentos[0]

Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Estrategias y Metodologías Pedagógicas', 'source': 'PDFs/Estrategias y Metodologías Pedagógicas.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Guía  Avanzada  de  Estrategias  Didácticas  \ny\n \nMetodologías\n \nPedagógicas\n \nEn  el  contexto  educativo  actual,  la  transición  de  un  modelo  de  enseñanza  centrado  en  el  \ndocente\n \na\n \nuno\n \ncentrado\n \nen\n \nel\n \nestudiante\n \nes\n \nfundamental.\n \nEste\n \ndocumento\n \ndetalla\n \nlas\n \nmetodologías\n \nmás\n \nefectivas,\n \nla\n \nintegración\n \nde\n \nhabilidades\n \ndel\n \nsiglo\n \nXXI\n \ny\n \nlas\n \ntendencias\n \nemergentes\n \nque\n \nestán\n \ntransformando\n \nlas\n \naulas.\n \n1.  Metodologías  Activas  de  Aprendizaje  \nEstas  estrategias  sitúan  al  estudiante  como  protagonista  de  su  propio  proceso  de  construcción  \ndel\n \nconocimiento.\n 

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    #Valores diferentes para un mayor contexto por el caso de uso para el docente
    separators=["\n\n","\n",". "," ",""]
)

fragmentos = divisor.split_documents(documentos)

fragmentos[7]

Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Estrategias y Metodologías Pedagógicas', 'source': 'PDFs/Estrategias y Metodologías Pedagógicas.pdf', 'total_pages': 3, 'page': 2, 'page_label': '3'}, page_content='Objetivos  de  Desarrollo  Sostenible  (ODS)  de  la  ONU  para  generar  conciencia  global.  \n5.  Principios  Transversales  para  la  Excelencia  Docente  \nSin  importar  la  metodología  elegida,  la  práctica  debe  regirse  por:  ●  Relevancia:  El  "para  qué"  debe  estar  siempre  claro.  ●  Feedback  (Retroalimentación):  Debe  ser  constante,  formativo  y  orientado  al  \ncrecimiento,\n \nno\n \nsolo\n \na\n \nla\n \ncalificación.\n ●  Seguridad  Psicológica:  Un  aula  donde  el  error  es  visto  como  una  oportunidad  de  \naprendizaje,\n \nno\n \ncomo\n \nun\n \nfracaso.\n ●  Flexibilidad:  La  capacidad  del  docente  para  pivotar  la  estrategia  si  el  grupo  lo  requiere.  Este  do

In [ ]:
!pip install -q langchain-google-genai faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 17.9 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore= FAISS.from_documents(
    documents=fragmentos,
    embedding=embeddings
)

vectorstore.index.reconstruct(0)

array([-0.00288102,  0.02389833,  0.01526948, ...,  0.00143856,
       -0.00374062,  0.00688161], dtype=float32)

In [ ]:
consulta="Cuales son los mejores métodos de enseñanza"

resultados = vectorstore.similarity_search(
    query=consulta,
    k=3
)

for i in resultados:
  print(i)
  print("--\n--")

page_content='●  Descripción:  Se  invierte  la  logística  tradicional:  la  instrucción  directa  ocurre  fuera  del  aula  
(videos,
 
lecturas)
 
y
 
el
 
tiempo
 
de
 
clase
 
se
 
reserva
 
para
 
la
 
aplicación
 
práctica
 
y
 
el
 
debate.
 ●  Efectividad:  Permite  una  mayor  personalización  del  aprendizaje  y  optimiza  el  rol  del  
docente
 
como
 
mentor
 
y
 
facilitador
 
en
 
lugar
 
de
 
simple
 
transmisor
 
de
 
datos.
 
1.4  Aprendizaje  Cooperativo  y  Colaborativo  
●  Descripción:  Estructuración  de  grupos  pequeños  donde  el  éxito  individual  depende  del  
éxito
 
del
 
grupo
 
(interdependencia
 
positiva).
 ●  Efectividad:  Mejora  las  competencias  socioemocionales,  la  empatía  y  la  capacidad  de  
negociación,
 
preparando
 
a
 
los
 
estudiantes
 
para
 
los
 
entornos
 
laborales
 
modernos.
 
1.5  Gamificación  (Gamification)' metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Estr

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

# Conexión con Google Sheets
En este apartado aplicaremos las configuraciones para extraer nuestra base de datos.

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# Iniciar sesión
auth.authenticate_user()

# Credenciales
creds, _ = default()
gc = gspread.authorize(creds)

# Nombre del archivo
hoja = gc.open('Alumnos').sheet1

# Obtener los datos
""" Estructura del archivo
      Estudiante | Nivel de Riesgo | Curso con mayor dificultad
      Alumno 1   |     Bajo        | Ingles
"""
# La primera fila siempre debe ser encabvezados - Devuelve la función una lista de diccionario
datos = hoja.get_all_records()

print(f"Estudiantes: {len(datos)} alumnos")
print(datos[0])

Estudiantes: 10 alumnos
{'Estudiante': 'Alumno 1', 'Nivel de Riesgo': 'Bajo', 'Curso con mayor dificultad': 'Ingles'}


# Clasificador de texto
Usaremos la IA para clasificar y tener una respuesta con mayor precisión dependiendo el contexto

In [ ]:
def clasificar_texto(question: str) -> str:
    prompt = f"""
    Eres un clasificador de rutas. Tienes 3 categorías para la consulta de un docente:
    1. "datos": La pregunta trata sobre notas, rendimiento, nivel de riesgo de estudiantes específicos.
    2. "local": La pregunta es sobre estrategias didácticas, metodologías pedagógicas o tendencias
    3. "ninguna": Si la pregunta no tiene ninguna relevancia en contexto educativo

    Consulta del docente: {question}

    Responde ÚNICAMENTE con la palabra "datos", "local","ninguna".
    """

    router_response = llm.invoke(prompt).content.strip().lower()

    if "datos" in router_response: return "datos"
    if "lical" in router_response: return "local"
    return "ninguna"


# Caso de Uso - Docente
Se requiere un asistente de consultas en el cual primero clasifica la información, luego en base al conocimiento proporciona el concepto y le brinda la respuesta al docente.

Para ello tenemos que construir el prompt y adaptar el contexto

In [ ]:
def asistente_responde(question: str) -> str:

    # Clasificamos el texto
    route = clasificar_texto(question)
    print(f"\nRuta: {route}\n")

    context = ""

    # Elegimos el contexto adecuado
    if route == "local":
        docs = retriever.invoke(question)
        context = "\n\n---\n\n".join(doc.page_content for doc in docs)

    elif route == "datos":
        context = "Datos de estudiantes\n"
        for row in datos:
            # Por si existe una celda vacia
            estudiante = row.get('Estudiante', 'Desconocido')
            riesgo = row.get('Nivel de Riesgo', 'No evaluado')
            dificultad = row.get('Curso con mayor dificultad', 'Ninguna')
            context += f"- Estudiante: {estudiante} | Nivel de Riesgo Predictivo: {riesgo} | Dificultad Principal: {dificultad}\n"


    # Construimos el prompt
    prompt = f"""Eres un asistente docente. Responde a las preguntas del docente basándote únicamente
    en el contexto proporcionado. Si la información no está en el contexto,
    di frontalmente que no tienes suficiente información de esa fuente y que debe consultar otra.

    Contexto: {context}

    Pregunta: {question}

    Respuesta (Responde siempre con profesionalismo y de forma pedagógica):"""

    # Devolver respuesta
    final_response = llm.invoke(prompt)
    return final_response.content

# Pruebas del modelo
Usaremos 3 casos de uso para evaluar las respuestas.

In [ ]:
asistente_responde("Cuales son los estudiantes con nivel de riesgo alto")


Ruta: datos



'Estimado docente,\n\nBasándome en los datos proporcionados, los estudiantes con un nivel de riesgo predictivo alto son:\n\n*   **Alumno 2**\n*   **Alumno 5**\n*   **Alumno 7**\n\nEspero que esta información le sea de utilidad para su planificación.'

In [ ]:
asistente_responde("Dime las 3 mejores metodologias para enseñar a mis estudiantes")


Ruta: local



'Estimado docente,\n\nEl documento "Guía Avanzada de Estrategias Didácticas y Metodologías Pedagógicas" detalla varias metodologías activas de aprendizaje y otras estrategias efectivas, describiendo sus beneficios y la forma en que sitúan al estudiante como protagonista.\n\nSin embargo, el contexto proporcionado no clasifica ni especifica cuáles son las "3 mejores" metodologías. Simplemente describe la efectividad de cada una de ellas (Aprendizaje Basado en Proyectos, Aprendizaje Basado en la Indagación, Aula Invertida, Aprendizaje Cooperativo y Colaborativo, Gamificación, Aprendizaje Personalizado y Diferenciado, Método Socrático, Metacognición y Aprendizaje Socioemocional).\n\nPara una clasificación o recomendación específica de las "3 mejores", deberá consultar otra fuente de información.'

In [ ]:
asistente_responde("Turquia es un buen pais para vivir?")


Ruta: ninguna



'Estimado/a docente,\n\nAgradezco su pregunta. Sin embargo, con el contexto que me ha proporcionado, no dispongo de información suficiente para determinar si Turquía es un buen país para vivir.\n\nPara obtener una respuesta a su consulta, le sugiero consultar fuentes de información adicionales que aborden aspectos como la calidad de vida, economía, seguridad, cultura y servicios en Turquía.\n\nQuedo a su disposición para cualquier otra pregunta que pueda responder con el contexto facilitado.'

# Recomendaciones y Mejoras
Este proyecto es una prueba que la IA clasifica correctamente y lo usamos para no sobrecargar tanto el contexto con datos que no puedan tener relevancia sin embargo también identifico oportunidades de mejoras para otra versión


*   Si el modelo clasifica como ninguno, se puede dejar un mensaje automático para no consumir los tokens del modelo para generar la respuesta
*   Se puede crear un apartado de combinado para que en relación de los estudiantes se pueda dar las mejores metodologías
* Si no encuentra la información, se dirige a la web (visto en la clase número 3)


El objetivo de este proyecto era aprovechar el ecosistema de Google ya que se pueden combinar soluciones, lo visto en clase + integrar una base de datos (Google Sheets), clasificar el texto y al ser exacto se disminuyen los errores del contexto ya que mientras más simple sea la información del modelo es mucho mejor.

Espero les haya gustado el proyecto y estoy abierto a nuevas recomendaciones
* Ig: https://www.instagram.com/22_yorman/
* Linkedin: https://www.linkedin.com/in/yormanjreyes/

